# CARec encoder experiments

- Run shared setup and function cells once, then run only the model sections you choose.
- Each variant has one fixed folder containing only its two best model checkpoints. Rerunning replaces them.
- Metrics stay in notebook outputs and memory. Automatic resume is not implemented.


## Shared settings

- Model and MLP run on MPS; sparse graph propagation and uniformity run on CPU.
- Settings below apply to every model. Set both epoch limits to 1 for a short check.


In [1]:
from pathlib import Path
from copy import copy, deepcopy
from time import perf_counter
import sys

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

PROJECT_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "requirements.txt").is_file()
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from benchmarks.carec import load_carec
from recbole.config import Config
from recbole.data.dataset import Dataset
from recbole.data.interaction import Interaction
from recbole.data.dataloader import FullSortEvalDataLoader
from recbole.sampler import Sampler
from recbole.evaluator import Collector, Evaluator
from recbole.utils import init_seed
from recbole.utils.enum_type import FeatureSource, FeatureType

carec = load_carec()

DATASET_ID = "electronics_carec_035_seed2023"
PREPARED_DIR = PROJECT_ROOT / "data/processed" / DATASET_ID
RECBOLE_ROOT = PROJECT_ROOT / "artifacts/recbole"
RECBOLE_DIR = RECBOLE_ROOT / DATASET_ID
EMBEDDING_DIR = PROJECT_ROOT / "artifacts/embeddings" / DATASET_ID
RUNS_DIR = PROJECT_ROOT / "artifacts/carec" / DATASET_ID

seed = 2023
phase1_epochs = 90
phase2_epochs = 90
patience = 50
weight_decay = 0.0
training_batch_size = 2048
learning_rate = 0.001

if not torch.backends.mps.is_available():
    raise RuntimeError("MPS is unavailable.")
device = torch.device("mps")


/Users/matouskovar/FIT/NI-PML/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Shared data

- Load interactions and RecBole ID mappings once. Keep the saved splits and repeated interactions.
- Padding has ID 0. Only training edges enter the model graph.


In [2]:
RECBOLE_DIR.mkdir(parents=True, exist_ok=True)
interaction_columns = ["user_id:token", "item_id:token", "timestamp:float"]
for split_name in ("train", "valid", "test", "cold"):
    split_path = RECBOLE_DIR / f"{DATASET_ID}.{split_name}.inter"
    if not split_path.exists():
        interactions = pd.read_csv(
            PREPARED_DIR / f"{split_name}.tsv.gz",
            sep="\t",
            usecols=interaction_columns,
            dtype={"user_id:token": str, "item_id:token": str},
            float_precision="round_trip",
        )
        # RecBole requires typed headers. Leave source_row out of this export.
        interactions[interaction_columns].to_csv(split_path, sep="\t", index=False)

catalogue_path = RECBOLE_DIR / f"{DATASET_ID}.item"
if not catalogue_path.exists():
    catalogue_ids = pd.read_csv(PREPARED_DIR / "item_ids.tsv.gz", sep="\t", dtype={"item_id": str})
    pd.DataFrame({"item_id:token": catalogue_ids["item_id"]}).to_csv(
        catalogue_path, sep="\t", index=False,
    )



In [3]:
dataset_settings = {
    "data_path": str(RECBOLE_ROOT),
    "benchmark_filename": ["train", "valid", "test", "cold"],
    "field_separator": "\t",
    "USER_ID_FIELD": "user_id",
    "ITEM_ID_FIELD": "item_id",
    "TIME_FIELD": "timestamp",
    "load_col": {
        "inter": ["user_id", "item_id", "timestamp"],
        "item": ["item_id"],
    },
    "normalize_all": False,
    "use_gpu": False,
}

base_config = Config(model=carec.CollabContex, dataset=DATASET_ID, config_dict=dataset_settings)
# Set before construction: CARec stores graph and item tensors outside registered buffers.
base_config["device"] = device
recbole_dataset = Dataset(base_config)
recbole_product_ids = recbole_dataset.field2id_token["item_id"]
train_dataset, valid_dataset, test_dataset, cold_dataset = recbole_dataset.build()

base_config["metrics"] = ["Recall", "NDCG"]
base_config["topk"] = [10, 50]
base_config["eval_batch_size"] = 64 * train_dataset.item_num


/Users/matouskovar/FIT/NI-PML/.venv/lib/python3.11/site-packages/recbole/data/dataset/dataset.py:638: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  feat[field].fillna(value=0, inplace=True)
/Users/matouskovar/FIT/NI-PML/.venv/lib/python3.11/site-packages/recbole/data/dataset/dataset.py:640: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on

In [4]:
# Upstream repeatable sampling leaves history empty. Supply cumulative history.
history_sampler = Sampler(
    ["train", "valid", "test", "cold"],
    [train_dataset, valid_dataset, test_dataset, cold_dataset],
)
evaluation_loaders = {}
for split_name, split in [("valid", valid_dataset), ("warm", test_dataset), ("cold", cold_dataset)]:
    phase = "test" if split_name == "warm" else split_name
    evaluation_loaders[split_name] = FullSortEvalDataLoader(
        base_config, split, history_sampler.set_phase(phase), shuffle=False,
    )

cold_product_ids = cold_dataset.inter_feat[base_config["ITEM_ID_FIELD"]].unique()
train_product_ids = train_dataset.inter_feat[base_config["ITEM_ID_FIELD"]].unique()
assert not set(cold_product_ids.tolist()) & set(train_product_ids.tolist())
print({name: len(loader.uid_list) for name, loader in evaluation_loaders.items()})


{'valid': 12386, 'warm': 81520, 'cold': 18680}


## Prepare one model

- Match cached vectors to RecBole IDs and add zero padding. Each variant gets separate item features and a fresh model.
- Keep original normalized edge weights. CPU sparse multiplication replaces edge-wise messages with differentiable transfers to MPS.
- Native dimensions also change user-table and MLP sizes. This is not yet a comparison at a common dimension.


In [5]:
def prepare_model(embedding_name):
    embedding_path = EMBEDDING_DIR / embedding_name
    with np.load(embedding_path, allow_pickle=False) as saved:
        item_ids = saved["item_ids"]
        vectors = saved["embeddings"]
        metadata = {
            "embedding_file": str(embedding_path.relative_to(PROJECT_ROOT)),
            "encoder": str(saved["model_id"]), "revision": str(saved["revision"]),
            "prompt": str(saved["prompt"]), "dimension": vectors.shape[1],
        }
    rows = {product_id: row for row, product_id in enumerate(item_ids)}
    assert len(rows) == len(item_ids), "Duplicate cache IDs"
    assert len(vectors) == len(item_ids), "Cache row count differs"
    assert recbole_product_ids[0] == "[PAD]"
    assert set(recbole_product_ids[1:]) == set(item_ids), "Product catalogues differ"
    aligned = np.zeros((train_dataset.item_num, vectors.shape[1]), dtype=np.float32)
    for item_id, product_id in enumerate(recbole_product_ids[1:], start=1):
        aligned[item_id] = vectors[rows[product_id]]

    config = deepcopy(base_config)
    dimension = metadata["dimension"]
    config["seed"] = seed
    config["encoder"] = "LightGCN"
    config["num_layers"] = 2
    config["embedding_size"] = dimension
    config["mlp"] = {
        "input_dim": dimension, "hidden_dims": [dimension, dimension],
        "output_dim": dimension, "num_layers": 2, "dropout": 0.2,
    }
    # Share immutable interactions; give this variant its own item features and metadata.
    model_dataset = copy(train_dataset)
    model_dataset.config = config
    for field in ("field2type", "field2source", "field2seqlen"):
        setattr(model_dataset, field, getattr(train_dataset, field).copy())
    model_dataset.set_field_property("item_emb", FeatureType.FLOAT_SEQ, FeatureSource.ITEM, dimension)
    model_dataset.item_feat = Interaction({
        **train_dataset.item_feat.interaction, "item_emb": torch.from_numpy(aligned),
    })
    init_seed(seed, config["reproducibility"])
    model = carec.CollabContex(config, model_dataset).to(device)
    encoder = model.encoder
    node_count = encoder.n_users + encoder.n_items
    adjacency = torch.sparse_coo_tensor(
        encoder.edge_index.flip(0).cpu(), encoder.edge_weight.cpu(), (node_count, node_count),
    ).coalesce().to_sparse_csr()

    def sparse_forward(node_vectors, edge_index, edge_weight):
        return torch.sparse.mm(adjacency, node_vectors.cpu()).to(node_vectors.device)

    def cpu_uniformity(vectors, t=2):
        return carec.CollabContex.uniformity(vectors.cpu(), t=t).to(vectors.device)

    encoder.gcn_conv.forward = sparse_forward
    model.uniformity = cpu_uniformity
    return model, config, metadata


## Evaluation

- RecBole Recall/NDCG at 10 and 50, averaged over users with positives. Warm ranks the full catalogue; cold ranks cold products with MLP bypassed.
- Mask padding and earlier history, except current positives. Repeated pairs across splits remain a benchmark limitation.


In [6]:
@torch.no_grad()
def evaluate_carec(model, loader, config, cold_only=False):
    collector = Collector(config)
    item_count = loader.dataset.item_num
    evaluator = Evaluator(config)
    previous_training = model.training
    previous_mlp_bypass = model.encoder.not_apply_mlp
    model.eval()
    if cold_only:
        model.encoder.not_apply_mlp = True
    # full_sort_predict caches graph outputs. Never reuse them across modes or epochs.
    model.restore_user_e = model.restore_item_e = None
    candidate_mask = torch.ones(item_count, dtype=torch.bool, device=config["device"])
    if cold_only:
        candidate_mask[:] = False
        cold_ids = loader.dataset.inter_feat[config["ITEM_ID_FIELD"]].unique()
        candidate_mask[cold_ids.to(config["device"])] = True
    candidate_mask[0] = False

    try:
        for interaction, history, positive_users, positive_items in tqdm(loader, desc="Cold" if cold_only else "Full ranking"):
            scores = model.full_sort_predict(interaction.to(config["device"]))
            scores = scores.view(-1, item_count)
            if not torch.isfinite(scores).all():
                raise ValueError("Non-finite recommendation scores before masking.")
            # Apply the item mask to columns, for every user in the batch.
            scores[:, ~candidate_mask] = -torch.inf
            if history is not None:
                scores[history[0].to(scores.device), history[1].to(scores.device)] = -torch.inf
            collector.eval_batch_collect(scores, interaction, positive_users, positive_items)
        return {name: float(value) for name, value in evaluator.evaluate(collector.get_data_struct()).items()}
    finally:
        model.encoder.not_apply_mlp = previous_mlp_bypass
        model.restore_user_e = model.restore_item_e = None
        model.train(previous_training)


## Train and save

- Phase 1 learns user vectors. Phase 2 freezes them and trains MLP, starting from the best phase-one weights.
- Validate every epoch; select by NDCG@10 and stop at the configured patience. Evaluate test splits only after training.
- Save only `phase1-best.pt` and `phase2-best.pt`, containing model weights. Rebuilding the model requires the same embedding cache and dataset ID mapping.


In [7]:
def train_phase(model, config, phase, max_epochs, run_dir):
    if max_epochs < 1 or patience < 1:
        raise ValueError("Epoch limit and patience must be positive.")
    if phase == 1:
        model.set_item_tutoring_phase()
        uniformity_field = config["USER_ID_FIELD"]
    elif phase == 2:
        model.set_user_tutoring_phase()
        uniformity_field = config["ITEM_ID_FIELD"]
    else:
        raise ValueError("CARec has phases 1 and 2.")

    # Frozen parameters have no optimizer state in this phase.
    model.zero_grad(set_to_none=True)
    optimizer = torch.optim.Adam(
        [parameter for parameter in model.parameters() if parameter.requires_grad],
        lr=learning_rate, weight_decay=weight_decay,
    )
    generator = torch.Generator().manual_seed(seed + phase)
    checkpoint_path = run_dir / f"phase{phase}-best.pt"
    best_score, stale_epochs = -float("inf"), 0
    best_epoch, best_metrics = None, None
    history = []
    started = perf_counter()

    for epoch in range(1, max_epochs + 1):
        model.train()
        indices = torch.randperm(len(train_dataset.inter_feat), generator=generator)
        total_loss, interaction_count = 0.0, 0
        for batch_ids in tqdm(indices.split(training_batch_size), desc=f"Phase {phase}, epoch {epoch}"):
            batch = train_dataset.inter_feat[batch_ids].to(config["device"])
            if batch[uniformity_field].unique().numel() < 2:
                raise ValueError("Uniformity needs at least two distinct active-side IDs.")
            optimizer.zero_grad(set_to_none=True)
            loss = model.calculate_loss(batch)
            if not torch.isfinite(loss):
                raise ValueError("Training loss is not finite.")
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * len(batch)
            interaction_count += len(batch)

        valid_metrics = evaluate_carec(model, evaluation_loaders["valid"], config)
        score = float(valid_metrics["ndcg@10"])
        if not np.isfinite(score):
            raise ValueError("Validation NDCG@10 is not finite.")
        improved = score > best_score
        stale_epochs = 0 if improved else stale_epochs + 1
        history.append({
            "epoch": epoch, "loss": total_loss / interaction_count,
            "valid": valid_metrics, "elapsed_seconds": perf_counter() - started,
        })
        if improved:
            best_score = score
            # Write a temporary file so interruption cannot truncate the previous best.
            temporary = checkpoint_path.with_suffix(".pt.tmp")
            best_epoch, best_metrics = epoch, valid_metrics
            torch.save(
                {name: value.detach().cpu() for name, value in model.state_dict().items()},
                temporary,
            )
            temporary.replace(checkpoint_path)
        print(f"Phase {phase}, epoch {epoch}: loss={history[-1]['loss']:.4f}, valid NDCG@10={score:.4f}, patience={stale_epochs}/{patience}")
        if stale_epochs >= patience:
            break

    # Reload the best weights before the next phase or final evaluation.
    best = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
    model.load_state_dict(best)
    model.restore_user_e = model.restore_item_e = None
    model.eval()
    return {"best_epoch": best_epoch, "valid": best_metrics, "epochs_run": len(history)}


In [8]:
experiment_results = {}


def run_experiment(embedding_name):
    variant = Path(embedding_name).stem
    run_dir = RUNS_DIR / variant
    checkpoints = sorted(path for path in run_dir.glob("*.pt") if path.is_file())
    if checkpoints:
        print(f"Skipping {embedding_name}: checkpoint exists ({', '.join(path.name for path in checkpoints)}).")
        print(f"Model directory: {run_dir}")
        return experiment_results.get(variant)

    started = perf_counter()
    model, config, metadata = prepare_model(embedding_name)
    run_dir.mkdir(parents=True, exist_ok=True)
    # Remove stale sidecars and temporary files before a fresh run.
    old_outputs = (
        "phase1-best.pt", "phase2-best.pt", "phase1-history.json", "phase2-history.json",
        "config.json", "id_maps.npz", "training.json", "results.json", "status.json",
    )
    for filename in old_outputs:
        (run_dir / filename).unlink(missing_ok=True)
        (run_dir / (filename + ".tmp")).unlink(missing_ok=True)
    experiment_results.pop(variant, None)
    print(f"{embedding_name}: {metadata['dimension']} dimensions, model {device}, graph CPU")
    print(f"Model directory: {run_dir}")

    training = {}
    training["phase1"] = train_phase(model, config, 1, phase1_epochs, run_dir)
    training["phase2"] = train_phase(model, config, 2, phase2_epochs, run_dir)
    test = {
        "warm": evaluate_carec(model, evaluation_loaders["warm"], config),
        "cold": evaluate_carec(model, evaluation_loaders["cold"], config, cold_only=True),
    }
    result = {
        "dimension": metadata["dimension"], "training": training, "test": test,
        "total_seconds": perf_counter() - started, "run_dir": str(run_dir),
    }
    experiment_results[variant] = result
    display(pd.DataFrame(test).T)
    return result


## Individual runs

- Run each cell independently after the shared cells above. Definitions alone do not start training.
- Returned values contain metrics and paths, not the trained model.


### MiniLM-L6-v2

- Input: without task instruction. The exact prompt is read from the cache and saved with the run.


In [9]:
minilm_result = run_experiment("all-MiniLM-L6-v2.npz")


/var/folders/wz/x7xqj0yj6ls_367syhygxjn40000gn/T/ipykernel_75025/822926853.py:45: UserWarning: Sparse CSR tensor support is in beta state. If you miss a functionality in the sparse tensor support, please submit a feature request to https://github.com/pytorch/pytorch/issues. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/SparseCsrTensorImpl.cpp:55.)
  ).coalesce().to_sparse_csr()


all-MiniLM-L6-v2.npz: 384 dimensions, model mps, graph CPU
Model directory: /Users/matouskovar/FIT/NI-PML/artifacts/carec/electronics_carec_035_seed2023/all-MiniLM-L6-v2


Full ranking: 100%|██████████| 194/194 [00:01<00:00, 100.32it/s]


Phase 1, epoch 1: loss=-2.5593, valid NDCG@10=0.0280, patience=0/50


Full ranking: 100%|██████████| 194/194 [00:01<00:00, 133.51it/s]


Phase 1, epoch 2: loss=-2.7233, valid NDCG@10=0.0309, patience=0/50


... (314 outputs left out)

Full ranking: 100%|██████████| 194/194 [00:01<00:00, 144.71it/s]


Phase 2, epoch 70: loss=-3.1863, valid NDCG@10=0.0452, patience=50/50


Cold: 100%|██████████| 292/292 [00:02<00:00, 99.15it/s] 


,recall@10,recall@50,ndcg@10,ndcg@50
warm,0.0714,0.1050,0.0619,0.0702
cold,0.0532,0.1344,0.0304,0.0488


### MPNet-base-v2

- Input: without task instruction. The exact prompt is read from the cache and saved with the run.


In [ ]:
mpnet_result = run_experiment("all-mpnet-base-v2.npz")


all-mpnet-base-v2.npz: 768 dimensions, model mps, graph CPU
Model directory: /Users/matouskovar/FIT/NI-PML/artifacts/carec/electronics_carec_035_seed2023/all-mpnet-base-v2


Full ranking: 100%|██████████| 194/194 [00:01<00:00, 111.12it/s]


Phase 1, epoch 1: loss=-2.5564, valid NDCG@10=0.0298, patience=0/50


Full ranking: 100%|██████████| 194/194 [00:01<00:00, 116.85it/s]


Phase 1, epoch 2: loss=-2.7485, valid NDCG@10=0.0334, patience=0/50


Full ranking: 100%|██████████| 194/194 [00:01<00:00, 115.66it/s]


... (226 outputs left out)

Phase 2, epoch 26: loss=-3.1909, valid NDCG@10=0.0467, patience=1/50


Full ranking: 100%|██████████| 194/194 [00:01<00:00, 130.93it/s]


Phase 2, epoch 27: loss=-3.1908, valid NDCG@10=0.0468, patience=2/50


Phase 2, epoch 28:  45%|████▌     | 100/221 [01:24<01:48,  1.11it/s]

### E5-base-v2

- Input: without task instruction, query prefix. The exact prompt is read from the cache and saved with the run.


In [ ]:
e5_base_result = run_experiment("e5-base-v2.npz")


### Qwen3-Embedding-0.6B

- Input: without task instruction. The exact prompt is read from the cache and saved with the run.


In [ ]:
qwen_plain_result = run_experiment("Qwen3-Embedding-0.6B-without_instruction.npz")


### Qwen3-Embedding-0.6B

- Input: with task instruction. The exact prompt is read from the cache and saved with the run.


In [ ]:
qwen_instruct_result = run_experiment("Qwen3-Embedding-0.6B-with_instruction.npz")


### Multilingual E5-large-instruct

- Input: without task instruction. The exact prompt is read from the cache and saved with the run.


In [ ]:
e5_large_plain_result = run_experiment("multilingual-e5-large-instruct-without_instruction.npz")


### Multilingual E5-large-instruct

- Input: with task instruction. The exact prompt is read from the cache and saved with the run.


In [ ]:
e5_large_instruct_result = run_experiment("multilingual-e5-large-instruct-with_instruction.npz")


### INSTRUCTOR-base

- Input: without task instruction. The exact prompt is read from the cache and saved with the run.


In [ ]:
instructor_plain_result = run_experiment("instructor-base-without_instruction.npz")


### INSTRUCTOR-base

- Input: with task instruction. The exact prompt is read from the cache and saved with the run.


In [ ]:
instructor_instruct_result = run_experiment("instructor-base-with_instruction.npz")


## Results from this session

- Show completed experiments held in memory. No result or configuration files are written.
- Restarting the kernel clears this table. Model checkpoints remain on disk.


In [ ]:
result_rows = []
for variant, result in experiment_results.items():
    row = {
        "variant": variant, "dimension": result["dimension"],
        "minutes": result["total_seconds"] / 60,
        "phase1_epoch": result["training"]["phase1"]["best_epoch"],
        "phase2_epoch": result["training"]["phase2"]["best_epoch"],
    }
    for split_name, metrics in result["test"].items():
        row.update({f"{split_name}_{metric}": value for metric, value in metrics.items()})
    result_rows.append(row)
display(pd.DataFrame(result_rows))
